In [1]:
import numpy as np
import pandas as pd
from nltk.stem.snowball import SnowballStemmer
from nltk.corpus import stopwords
import re
import ftfy
import html
pd.set_option('display.max_colwidth', None)

In [2]:
df=pd.read_csv("development.csv",delimiter=",", index_col="Id")

### *Source* feature inspection

In [3]:
n_nan_source = df['source'].isna().sum()
n_empty_soruce = df['source'].astype(str).str.strip().eq('').sum()
n_placeholders_source = df['source'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_source}")
print(f"Number of empty rows: {n_empty_soruce}")
print(f"Number of placeholders (\\N): {n_placeholders_source}")
df['source'] = df['source'].replace('\\N', 'Unknown')
source_counts = df['source'].value_counts()
selected_sources = source_counts[source_counts >= 50].index.to_list()
selected_sources.remove('Unknown')
print(f"Relevant Sources:\n{selected_sources}")
coverage = source_counts[selected_sources].sum() / len(df)
print(f"Number of selected sources: {len(selected_sources)}")
print(f"Percentage of selected sources: {coverage:.2%}")

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 294
Relevant Sources:
['Yahoo', 'Reuters', 'BBC', 'New', 'Washington', 'RedNova', 'Boston', 'CNN', 'CNET', 'Topix.Net', 'Guardian', 'Motley', 'Register', 'International', 'Forbes', 'Time', 'ABC', 'InfoWorld', 'San', 'Wired', 'Xinhua', 'Computerworld', 'News', 'CSMonitor', 'PCWorld', 'Bloomberg', 'Seattle', 'Ananova', 'Syfy.com', 'Voice', 'USA', 'Independent', 'Scotsman', 'CBS', 'Rediff', 'Times', 'Channel', 'CBC', 'Newsday', 'Newsweek', 'Houston', 'Australian', 'Daily', 'Telegraph.co.uk', 'ESPN', 'Canada.com', 'BCC', 'Sports', 'Search', 'Chicago', 'Turkish', 'CNN/SI', 'MSNBC', 'London', 'National', 'Financial', 'Toronto', 'Indianapolis', 'Melbourne', 'Christian', 'Detroit', 'ZDNet.com', 'CTV', 'PC', 'ic', 'NEWS.com.au', 'RTE', 'Scotland', 'Hindustan', 'NPR', 'Al-Jazeera', 'Information', 'IPS', 'TechNewsWorld', 'News24', 'sportinglife.com', 'Arizona', 'Age', 'Taipei', 'Radio', 'Gulf', 'Sun-Sentinel.com', 'Indian'

### *Title* feature inspection

In [4]:
n_nan_title = df['title'].isna().sum()
n_empty_title = df['title'].astype(str).str.strip().eq('').sum()
n_placeholders_title = df['title'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_title}")
print(f"Number of empty rows: {n_empty_title}")
print(f"Number of placeholders (\\N): {n_placeholders_title}")
print("Titles Sample:")
print(df['title'].sample(20))

Number of NaN rows: 1
Number of empty rows: 2
Number of placeholders (\N): 0
Titles Sample:
Id
2561                                       Tesco challenges iTunes&#39; domination
34408                                 Former Bishop Indicted On Child Rape Charges
64905                                               Unbeaten Norton stops Westwood
32334                                                         Woods plays catch up
34690                                    Co. Pulls Toys Depicting 9-11 Attack (AP)
23277                                             German nurse jailed for killings
42516                                               Muslim leaders condemn killers
9964                                             Thai coup chiefs seek civilian PM
17088                     Three Car Bombs Explode Across Iraq, Killing at Least 26
73873                         Friend names suspect in spy poisoning \\n    (AP)\\n
35505                                      Derry rail link group keeps pres

### *Article* feature inspection

In [5]:
n_nan_article = df['article'].isna().sum()
n_empty_article = df['article'].astype(str).str.strip().eq('').sum()
n_placeholders_article = df['article'].astype(str).str.strip().eq('\\N').sum()
print(f"Number of NaN rows: {n_nan_article}")
print(f"Number of empty rows: {n_empty_article}")
print(f"Number of placeholders (\\N): {n_placeholders_article}")
print("Articles Sample")
print(df['article'].sample(10))

Number of NaN rows: 1
Number of empty rows: 7
Number of placeholders (\N): 1874
Articles Sample
Id
21717                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                             Israel&#39;s threats against Syria after the Beersheba suicide bombings would &quot;exacerbate the deteriorating situation in the region,&quot; the Syrian foreign minister said ye

### *PageRank* feature inspection

In [6]:
n_nan_pr = df['page_rank'].isna().sum()
n_empty_pr = df['page_rank'].astype(str).str.strip().eq('').sum()
n_placeholders_pr = df['page_rank'].astype(str).str.strip().eq('\\N').sum()
rank_5=np.array([df['page_rank'].values==5]).sum()
print(f"Number of NaN rows: {n_nan_pr}")
print(f"Number of empty rows: {n_empty_pr}")
print(f"Number of placeholders (\\N): {n_placeholders_pr}")
print(f"Number of articles with PageRank 5: {rank_5}")
print(df['page_rank'].sample(10))

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 0
Number of articles with PageRank 5: 73891
Id
78003    5
27085    5
41612    5
61069    5
67450    5
35452    5
29942    5
2595     5
31185    5
57394    5
Name: page_rank, dtype: int64


### *Timestamp* feature inspection 

In [7]:
n_nan_time = df['timestamp'].isna().sum()
n_empty_time = df['timestamp'].astype(str).str.strip().eq('').sum()
n_placeholders_time = df['timestamp'].astype(str).str.strip().eq('\\N').sum()
n_uslesess_time=np.array([df['timestamp'].values=="0000-00-00 00:00:00"]).sum()
print(f"Number of NaN rows: {n_nan_time}")
print(f"Number of empty rows: {n_empty_time}")
print(f"Number of placeholders (\\N): {n_placeholders_time}")
print(f"Number invalid dates (0000-00-00 00:00:00): {n_uslesess_time}")
print(df['timestamp'].sample(10))

Number of NaN rows: 0
Number of empty rows: 0
Number of placeholders (\N): 0
Number invalid dates (0000-00-00 00:00:00): 27750
Id
57031    2007-08-24 18:01:14
19813    2007-07-24 13:08:11
258      2007-09-13 18:15:48
962      2008-01-02 07:25:01
47239    2007-02-09 20:27:08
62027    0000-00-00 00:00:00
8512     0000-00-00 00:00:00
40967    2006-09-24 17:53:44
71491    2006-12-14 20:57:26
49112    0000-00-00 00:00:00
Name: timestamp, dtype: object


### *Timestamp* feature processing

In [8]:
def process_timestamp(df):
    df = df.copy()
    
    df['dt_obj'] = pd.to_datetime(df['timestamp'], errors='coerce')
    df['has_date'] = df['dt_obj'].notna().astype(int)
    
    df['year'] = df['dt_obj'].dt.year.fillna(-1).astype(int)
    df['month'] = df['dt_obj'].dt.month.fillna(-1).astype(int)
    df['day_of_week'] = df['dt_obj'].dt.dayofweek.fillna(-1).astype(int)
    
    hours = df['dt_obj'].dt.hour
    df['time_of_day'] = pd.cut(hours, bins=[0, 6, 12, 18, 24], labels=[0, 1, 2, 3], right=False) 
    
    df['time_of_day'] = df['time_of_day'].astype('float').fillna(-1).astype(int)
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    
    df = df.drop(columns=['timestamp', 'dt_obj'])
    
    return df

df = process_timestamp(df)

new_cols = ['has_date', 'year', 'month', 'day_of_week', 'time_of_day', 'is_weekend']
print(f"Nuove colonne aggiunte: {new_cols}")
print("Sample of 10 timestamps:")
print(df[new_cols].sample(10))

Nuove colonne aggiunte: ['has_date', 'year', 'month', 'day_of_week', 'time_of_day', 'is_weekend']
Sample of 10 timestamps:
       has_date  year  month  day_of_week  time_of_day  is_weekend
Id                                                                
63142         1  2007      7            4            1           0
47383         1  2007     12            5            0           1
75700         0    -1     -1           -1           -1           0
1988          0    -1     -1           -1           -1           0
14512         1  2008      1            3            3           0
51343         1  2004     12            5            2           1
67277         1  2004     10            5            0           1
64605         0    -1     -1           -1           -1           0
54141         1  2004     12            3            3           0
32730         1  2008      2            4            1           0


### *Title* feature stemming

In [9]:
import nltk
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_title(text):
    if pd.isna(text) or text == "": return ""
    text = str(text)
    
    text = html.unescape(text)
    text = ftfy.fix_text(text)
    text = text.lower()
    text = text.replace('.', '') 
    
    text = re.sub(r'(?:\\n|\s)*\(.*?\)\W*$', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)

    money_pattern = r'([$£€]\s*\d+(\.\d+)?)|(\d+(\.\d+)?\s+(m|bn|k|t)\b)|(\d+(\.\d+)?\s*(bn|k|t)\b)|(\d+(\.\d+)?\s*(dollar|euro|pound))'
    text = re.sub(money_pattern, ' tag_money ', text)

    percent_pattern = r'\d+(\.\d+)?\s*(%|percent|pct)'
    text = re.sub(percent_pattern, ' tag_percent ', text)

    score_pattern = r'\b\d{1,3}\s*-\s*\d{1,3}\b'
    text = re.sub(score_pattern, ' tag_score ', text)
    
    year_pattern = r'\b(19|20)\d{2}\b'
    text = re.sub(year_pattern, ' tag_year ', text)

    #  togliamo solo la punteggiatura pura.
    text = re.sub(r'[^a-z0-9]', ' ', text)

    words = text.split()
    
    meaningful_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) >= 2 and not w.isnumeric()]
    
    return " ".join(meaningful_words)

# Test rapido
def test_title(titles_series, cleaner_func, n):
    sample = titles_series.sample(n)
    print(f"Test on {n} random titles\n")
    for _, text in sample.items():
        cleaned = cleaner_func(text)
        print(f"Original text: {text}")
        print(f"Processed text: {cleaned}")
        print("-" * 50)

test_title(df['title'], clean_title, n=10)

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/giorgiozoccatelli/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Test on 10 random titles

Original text: SBC Reports Lower Fourth-Quarter Earnings
Processed text: sbc report lower fourth quarter earnings
--------------------------------------------------
Original text: Solar firm's focus: power to the people
Processed text: solar firm focus power people
--------------------------------------------------
Original text: Delta to offer HBO programs
Processed text: delta offer hbo program
--------------------------------------------------
Original text: Iraqi rebels release 10 Turkish hostages
Processed text: iraqi rebel release turkish hostage
--------------------------------------------------
Original text: G8 leaders strike deal on Africa pledge: diplomat
Processed text: g8 leader strike deal africa pledge diplomat
--------------------------------------------------
Original text: Bush Calls Out Wolves on Kerry
Processed text: bush call wolf kerry
--------------------------------------------------
Original text: DealBook Blog: Lufthansa Prepares to B

### *Article* feature stemming

In [10]:
def clean_article(text):
    if pd.isna(text) or text == "" or str(text).strip() == "\\N": 
        return ""
    text = str(text)

    text = text[:1000] 

    text = html.unescape(text)
    text = re.sub(r'http[s]?://\S+', ' ', text)
    text = re.sub(r'\b[a-z0-9]+\.(com|net|org|gov)\b', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    
    trash_pattern = r'\b(src|href|alt|width|height|align|border|style|sig|valign|hspace|vspace)\b'
    text = re.sub(trash_pattern, ' ', text, flags=re.IGNORECASE)
    
    text = re.sub(r'\b[a-z]*\d{3,}[a-z]*\b', ' ', text)
    text = re.sub(r'\b[bcdfghjklmnpqrstvwxyz]{4,}\b', ' ', text) 

    text = ftfy.fix_text(text)
    text = text.lower()
    text = text.strip()
    
    text = re.sub(r'^\s*[a-z][\w\s,\.\(\)]{0,50}\s*--\s*', '', text)
    text = re.sub(r'^\s*[a-z][^\.\?!]{2,50}\s+[-–—]\s+', '', text)
    
    agencies_pattern = r'(?i)^\s*.*?\b(reuters|afp|ap|upi|bloomberg|bbc|cnn|blog)\b.*?\s*[-:–—]\s*'
    text = re.sub(agencies_pattern, '', text)

    text = re.sub(r'(?i)^by\s+[a-z\s\.,]+\s{2,}', '', text)
    text = text.replace('.', '')

    money_pattern = r'([$£€]\s*\d+(\.\d+)?)|(\d+(\.\d+)?\s+(m|bn|k|t)\b)|(\d+(\.\d+)?\s*(bn|k|t)\b)|(\d+(\.\d+)?\s*(dollar|euro|pound))'
    text = re.sub(money_pattern, ' tag_money ', text)

    percent_pattern = r'\d+(\.\d+)?\s*(%|percent|pct)'
    text = re.sub(percent_pattern, ' tag_percent ', text)

    score_pattern = r'\b\d{1,3}\s*-\s*\d{1,3}\b'
    text = re.sub(score_pattern, ' tag_score ', text)
    
    year_pattern = r'\b(19|20)\d{2}\b'
    text = re.sub(year_pattern, ' tag_year ', text)

    text = re.sub(r'[^a-z0-9]', ' ', text)
    words = text.split()
    
    meaningful_words = [lemmatizer.lemmatize(w) for w in words if w not in stop_words and len(w) >= 2 and not w.isnumeric()]
    
    return " ".join(meaningful_words)

# Test
def test_article(article_series, cleaner_func, n):
    sample = article_series.sample(n)
    print(f"Test on {n} random articles\n")
    for _, text in sample.items():
        print(f"Original text (First 200 char): {str(text)}") 
        print(f"Processed text: {cleaner_func(text)}")
        print("-" * 50)

test_article(df['article'], clean_article, n=100)

Test on 100 random articles

Original text (First 200 char): An MP launches a campaign to allow disabled children under three to qualify for mobility allowance.
Processed text: mp launch campaign allow disabled child three qualify mobility allowance
--------------------------------------------------
Original text (First 200 char): Singer Rod Stewart has gone to the top of the US album chart for the first time in 25 years.
Processed text: singer rod stewart gone top u album chart first time year
--------------------------------------------------
Original text (First 200 char): TORONTO (Reuters) - People eat more when they are glued to the television, and the more entertaining the program, the more they eat, according to research presented on Saturday.
Processed text: people eat glued television entertaining program eat according research presented saturday
--------------------------------------------------
Original text (First 200 char): This week, NASA has concerned itself with the fac

### *Title + Article* features merge

In [11]:
print(f"Starting shape: {df.shape}")
print(f"Starting columns: {df.columns.tolist()}")
df['title_clean'] = df['title'].apply(clean_title)
df['article_clean'] = df['article'].apply(clean_article)
df['text_combined'] = (df['title_clean'] + " " + df['title_clean'] + " " + df['article_clean']).str.strip()

n_empty = (df['text_combined'] == "").sum()
print(f"Removing {n_empty} rows with empty text")
df = df[df['text_combined'] != ""]
df = df.drop(columns=['title', 'article', 'title_clean', 'article_clean'])

print(f"Final shape: {df.shape}")
print(f"Actual columns: {df.columns.tolist()}")
print("Example of title + article combined:")
print(df['text_combined'].iloc[0])

Starting shape: (79997, 11)
Starting columns: ['source', 'title', 'article', 'page_rank', 'label', 'has_date', 'year', 'month', 'day_of_week', 'time_of_day', 'is_weekend']
Removing 3 rows with empty text
Final shape: (79994, 10)
Actual columns: ['source', 'page_rank', 'label', 'has_date', 'year', 'month', 'day_of_week', 'time_of_day', 'is_weekend', 'text_combined']
Example of title + article combined:
opec boost nigeria oil revenue 82m bpd opec boost nigeria oil revenue 82m bpd organisation petroleum exporting country opec hiking official output one million barrel per day effective november nigeria getting barrel per day per cent new quota


### Encoding categorical features

In [12]:
df['source'] = np.where(df['source'].isin(selected_sources), df['source'], 'Other')
categorical_cols = ['source']

print(f"Shape before encoding: {df.shape}")

df = pd.get_dummies(
    df, 
    columns=categorical_cols, 
    prefix=categorical_cols, 
    prefix_sep='_', 
    dtype=int
)

print(f"Shape after encoding: {df.shape}")
source_cols = [c for c in df.columns if c.startswith('source_')]
print(f"Number of generated features: {len(source_cols)}")

Shape before encoding: (79994, 10)
Shape after encoding: (79994, 109)
Number of generated features: 100


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from scipy.sparse import hstack 

target_col = 'label' 
custom_stop_words = [
    'said','say' ,'says','report', 'reported','according', 'today', 'yesterday', 'tomorrow', 'week', 'month', 'day','year', 'years', 'time','read', 'story', 'latest', 'images', 'click', 'link', 'news', 'press']
my_stop_words = list(ENGLISH_STOP_WORDS) + custom_stop_words

y = df[target_col]
X = df.drop(columns=[target_col])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train set: {X_train.shape}")
print(f"Test set:  {X_test.shape}")

tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.7,
    stop_words=my_stop_words,
    sublinear_tf=True,
    norm='l2'
)

X_train_vectorized = tfidf.fit_transform(X_train['text_combined'])
X_test_vectorized = tfidf.transform(X_test['text_combined'])

X_train_base = X_train.drop(columns=['text_combined'])
X_test_base = X_test.drop(columns=['text_combined'])

X_train_final = hstack([X_train_vectorized, X_train_base])
X_test_final = hstack([X_test_vectorized, X_test_base])

print(f"Matrix Train: {X_train_final.shape}")
print(f"Matrix Test:  {X_test_final.shape}")

Train set: (63995, 108)
Test set:  (15999, 108)
Matrix Train: (63995, 30107)
Matrix Test:  (15999, 30107)


In [17]:
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score
import numpy as np
import time

# --- STEP 1: PREPARAZIONE DATI PER XGBOOST (LSA) ---
# XGBoost è lento su 30k colonne. Usiamo TruncatedSVD per comprimere SOLO la parte testuale
# Input: X_train_vectorized (che hai creato nella cella precedente)
print("1. Esecuzione LSA (SVD) per ridurre le dimensioni per XGBoost...")
svd = TruncatedSVD(n_components=400, random_state=42)

# Riduciamo solo la parte di testo vettorizzato
X_train_svd = svd.fit_transform(X_train_vectorized)
X_test_svd = svd.transform(X_test_vectorized)

print(f"   Dimensioni testo ridotte da {X_train_vectorized.shape[1]} a {X_train_svd.shape[1]}")

# Creiamo la matrice densa per XGBoost unendo SVD + Feature Base
# X_train_base nella tua cella precedente è un DataFrame, prendiamo i .values
X_train_lsa = np.hstack([X_train_svd, X_train_base.values])
X_test_lsa = np.hstack([X_test_svd, X_test_base.values])

print(f"   Shape finale Input XGBoost (Denso): {X_train_lsa.shape}")


# --- STEP 2: DEFINIZIONE DEI MODELLI ---

# Modello A: Logistic Regression (Lavora sulla matrice SPARSA COMPLETA originale)
# Usa X_train_final che hai creato nella cella precedente
log_reg = LogisticRegression(
    C=1.0, 
    solver='saga', 
    multi_class='multinomial',
    class_weight='balanced', # Aiuta le classi deboli
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

# Modello B: XGBoost (Lavora sulla matrice DENSA RIDOTTA appena creata)
# Parametri ottimizzati per velocità e performance su dati densi
xgb_lsa = XGBClassifier(
    n_estimators=1500,
    learning_rate=0.03,
    max_depth=8,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.6,
    objective='multi:softprob',
    num_class=len(y.unique()),
    n_jobs=-1,
    random_state=42,
    tree_method='hist',      # Fondamentale per la velocità
    early_stopping_rounds=100
)


# --- STEP 3: TRAINING ---
print("\n2. Avvio Training Modelli...")

# Train Logistic Regression
start_lr = time.time()
print("   Training Logistic Regression (su 30k feature sparse)...")
log_reg.fit(X_train_final, y_train)
print(f"   LR Completata in {(time.time() - start_lr):.1f} sec.")

# Train XGBoost
start_xgb = time.time()
print("   Training XGBoost (su ~500 feature dense)...")
xgb_lsa.fit(
    X_train_lsa, y_train,
    eval_set=[(X_test_lsa, y_test)],
    verbose=0 # Silenzioso per pulizia output
)
print(f"   XGB Completato in {(time.time() - start_xgb)/60:.1f} min.")


# --- STEP 4: ENSEMBLE (MEDIA PESATA) ---
print("\n3. Calcolo Ensemble...")

# Otteniamo le probabilità da entrambi i modelli
# LR usa X_test_final (quello della tua cella)
proba_lr = log_reg.predict_proba(X_test_final)

# XGB usa X_test_lsa (quello ridotto creato qui sopra)
proba_xgb = xgb_lsa.predict_proba(X_test_lsa)

# Media pesata (50% LR, 50% XGB)
# Puoi sbilanciare se uno dei due è molto più forte (es. 0.4 LR, 0.6 XGB)
proba_ensemble = (0.5 * proba_lr) + (0.5 * proba_xgb)

# Scelta della classe finale
y_pred_ensemble = np.argmax(proba_ensemble, axis=1)


# --- STEP 5: RISULTATI FINALI ---
print("\n--- RISULTATI ENSEMBLE (Logistic Regression + XGBoost/LSA) ---")
print(classification_report(y_test, y_pred_ensemble))
print(f"MACRO F1 SCORE: {f1_score(y_test, y_pred_ensemble, average='macro'):.4f}")

1. Esecuzione LSA (SVD) per ridurre le dimensioni per XGBoost...
   Dimensioni testo ridotte da 30000 a 400
   Shape finale Input XGBoost (Denso): (63995, 507)

2. Avvio Training Modelli...
   Training Logistic Regression (su 30k feature sparse)...


/Users/giorgiozoccatelli/miniforge3/envs/data/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
/Users/giorgiozoccatelli/miniforge3/envs/data/lib/python3.10/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


   LR Completata in 44.9 sec.
   Training XGBoost (su ~500 feature dense)...
   XGB Completato in 4.7 min.

3. Calcolo Ensemble...

--- RISULTATI ENSEMBLE (Logistic Regression + XGBoost/LSA) ---
              precision    recall  f1-score   support

           0       0.69      0.80      0.74      4708
           1       0.74      0.79      0.76      2118
           2       0.83      0.79      0.81      2232
           3       0.62      0.43      0.50      1995
           4       0.77      0.88      0.82      1715
           5       0.56      0.46      0.50      2611
           6       0.63      0.65      0.64       620

    accuracy                           0.70     15999
   macro avg       0.69      0.68      0.68     15999
weighted avg       0.69      0.70      0.69     15999

MACRO F1 SCORE: 0.6830
